In [7]:
# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths
DATA_PATH = Path("../data/processed/cleaned_stock_data.csv")
OUTPUT_PATH = Path("../data/processed/feature_engineered_data.csv")

# Load dataset
stocks_df = pd.read_csv(DATA_PATH)

# Convert Date
stocks_df["Date"] = pd.to_datetime(stocks_df["Date"])

# Sort chronologically stock-wise
stocks_df = (
    stocks_df
    .sort_values(["Symbol", "Date"])
    .reset_index(drop=True)
)

# Basic validation
print("Dataset Shape:", stocks_df.shape)
print("Unique Stocks:", stocks_df["Symbol"].nunique())
print("Date Range:", stocks_df["Date"].min(), "to", stocks_df["Date"].max())

stocks_df.head()

Dataset Shape: (235192, 18)
Unique Stocks: 49
Date Range: 2000-01-03 00:00:00 to 2021-04-30 00:00:00


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,%Deliverble,Company Name,Industry,ISIN Code
0,2007-11-27,ADANIPORTS,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,27294366,2.687719e+15,NaN,9859619.0,0.3612,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
1,2007-11-28,ADANIPORTS,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,4581338,4.312765e+14,NaN,1453278.0,0.3172,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
2,2007-11-29,ADANIPORTS,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,5124121,4.550658e+14,NaN,1069678.0,0.2088,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
3,2007-11-30,ADANIPORTS,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,4609762,4.283257e+14,NaN,1260913.0,0.2735,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042
4,2007-12-03,ADANIPORTS,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,2977470,2.875200e+14,NaN,816123.0,0.2741,Adani Ports and Special Economic Zone Ltd.,SERVICES,INE742F01042


In [8]:
# ==========================================
# 2. RETURN, LAG, TREND & MOMENTUM FEATURES
# ==========================================

# ------------------------------------------
# A. Return Features
# ------------------------------------------

stocks_df["Daily_Return"] = (
    stocks_df.groupby("Symbol")["Close"]
    .pct_change()
)

stocks_df["Log_Return"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(lambda x: np.log(x / x.shift(1)))
)

stocks_df["Intraday_Return"] = (
    (stocks_df["Close"] - stocks_df["Open"])
    / stocks_df["Open"]
)

stocks_df["High_Low_Range"] = (
    (stocks_df["High"] - stocks_df["Low"])
    / stocks_df["Open"]
)


# ------------------------------------------
# B. Lag Features
# ------------------------------------------

for lag in [1, 2, 3, 5, 10]:

    stocks_df[f"Return_Lag_{lag}"] = (
        stocks_df.groupby("Symbol")["Daily_Return"]
        .shift(lag)
    )


for lag in [1, 2, 3, 5]:

    stocks_df[f"Close_Lag_{lag}"] = (
        stocks_df.groupby("Symbol")["Close"]
        .shift(lag)
    )


# ------------------------------------------
# C. Simple Moving Averages
# ------------------------------------------

sma_windows = [5, 10, 20, 50, 100, 200]

for window in sma_windows:

    stocks_df[f"SMA_{window}"] = (
        stocks_df.groupby("Symbol")["Close"]
        .transform(
            lambda x: x.rolling(window).mean()
        )
    )

    stocks_df[f"Price_SMA_{window}_Ratio"] = (
        stocks_df["Close"]
        / stocks_df[f"SMA_{window}"]
    )


# ------------------------------------------
# D. Exponential Moving Averages
# ------------------------------------------

stocks_df["EMA_12"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.ewm(span=12, adjust=False).mean()
    )
)

stocks_df["EMA_26"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.ewm(span=26, adjust=False).mean()
    )
)


# ------------------------------------------
# E. Momentum
# ------------------------------------------

for period in [5, 10, 20]:

    stocks_df[f"Momentum_{period}"] = (
        stocks_df.groupby("Symbol")["Close"]
        .pct_change(period)
    )


# ------------------------------------------
# F. RSI
# ------------------------------------------

def calculate_rsi(series, window=14):

    delta = series.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()

    rs = avg_gain / avg_loss

    return 100 - (100 / (1 + rs))


stocks_df["RSI_14"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(calculate_rsi)
)


# ------------------------------------------
# G. MACD
# ------------------------------------------

stocks_df["MACD"] = (
    stocks_df["EMA_12"]
    - stocks_df["EMA_26"]
)

stocks_df["MACD_Signal"] = (
    stocks_df.groupby("Symbol")["MACD"]
    .transform(
        lambda x: x.ewm(span=9, adjust=False).mean()
    )
)

stocks_df["MACD_Histogram"] = (
    stocks_df["MACD"]
    - stocks_df["MACD_Signal"]
)


print("Return, lag, trend and momentum features created.")

stocks_df.head()

Return, lag, trend and momentum features created.


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,Price_SMA_200_Ratio,EMA_12,EMA_26,Momentum_5,Momentum_10,Momentum_20,RSI_14,MACD,MACD_Signal,MACD_Histogram
0,2007-11-27,ADANIPORTS,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,...,NaN,962.900000,962.900000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
1,2007-11-28,ADANIPORTS,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,...,NaN,952.284615,957.788889,NaN,NaN,NaN,NaN,-5.504274,-1.100855,-4.403419
2,2007-11-29,ADANIPORTS,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,...,NaN,941.810059,952.337860,NaN,NaN,NaN,NaN,-10.527801,-2.986244,-7.541557
3,2007-11-30,ADANIPORTS,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,...,NaN,938.693127,950.057278,NaN,NaN,NaN,NaN,-11.364151,-4.661825,-6.702326
4,2007-12-03,ADANIPORTS,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,...,NaN,943.401877,951.482665,NaN,NaN,NaN,NaN,-8.080788,-5.345618,-2.735170


In [9]:
# ==========================================
# 3. VOLATILITY, VOLUME & CALENDAR FEATURES
# ==========================================

# ------------------------------------------
# A. Bollinger Bands
# ------------------------------------------

stocks_df["BB_Middle"] = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

bb_std = (
    stocks_df.groupby("Symbol")["Close"]
    .transform(
        lambda x: x.rolling(20).std()
    )
)

stocks_df["BB_Upper"] = (
    stocks_df["BB_Middle"] + 2 * bb_std
)

stocks_df["BB_Lower"] = (
    stocks_df["BB_Middle"] - 2 * bb_std
)

stocks_df["BB_Position"] = (
    (stocks_df["Close"] - stocks_df["BB_Lower"])
    /
    (stocks_df["BB_Upper"] - stocks_df["BB_Lower"])
)


# ------------------------------------------
# B. Rolling Volatility
# ------------------------------------------

for window in [5, 10, 20, 30]:

    stocks_df[f"Volatility_{window}"] = (
        stocks_df.groupby("Symbol")["Daily_Return"]
        .transform(
            lambda x: x.rolling(window).std()
        )
    )


# ------------------------------------------
# C. Volume Features
# ------------------------------------------

stocks_df["Volume_Change"] = (
    stocks_df.groupby("Symbol")["Volume"]
    .pct_change()
)

stocks_df["Volume_MA_20"] = (
    stocks_df.groupby("Symbol")["Volume"]
    .transform(
        lambda x: x.rolling(20).mean()
    )
)

stocks_df["Relative_Volume"] = (
    stocks_df["Volume"]
    / stocks_df["Volume_MA_20"]
)


# ------------------------------------------
# D. Calendar Features
# ------------------------------------------

stocks_df["Year"] = stocks_df["Date"].dt.year
stocks_df["Month"] = stocks_df["Date"].dt.month
stocks_df["DayOfWeek"] = stocks_df["Date"].dt.dayofweek
stocks_df["Quarter"] = stocks_df["Date"].dt.quarter


print("Volatility, volume and calendar features created.")

Volatility, volume and calendar features created.


In [10]:
# ==========================================
# 4. TARGET CREATION, VALIDATION & SAVE
# ==========================================

# ------------------------------------------
# A. Future 1-Day Return
# ------------------------------------------

stocks_df["Future_Return_1D"] = (
    stocks_df.groupby("Symbol")["Close"]
    .shift(-1)
    / stocks_df["Close"]
    - 1
)


# ------------------------------------------
# B. Direction Target
# ------------------------------------------

stocks_df["Target_Direction"] = np.where(
    stocks_df["Future_Return_1D"] > 0,
    1,
    0
)

# Last observation of every stock has no future return.
# Do NOT incorrectly classify it as DOWN.

stocks_df.loc[
    stocks_df["Future_Return_1D"].isna(),
    "Target_Direction"
] = np.nan


# ------------------------------------------
# C. Replace Infinite Values
# ------------------------------------------

numeric_cols = stocks_df.select_dtypes(
    include=np.number
).columns

stocks_df[numeric_cols] = (
    stocks_df[numeric_cols]
    .replace([np.inf, -np.inf], np.nan)
)


# ------------------------------------------
# D. Missing Value Inspection
# ------------------------------------------

missing_features = (
    stocks_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print("\nColumns containing missing values:\n")

print(
    missing_features[
        missing_features > 0
    ]
)


# ------------------------------------------
# E. Target Distribution
# ------------------------------------------

print("\nTarget Distribution:")

print(
    stocks_df["Target_Direction"]
    .value_counts(dropna=False)
)

print("\nTarget Distribution (%):")

print(
    stocks_df["Target_Direction"]
    .value_counts(normalize=True, dropna=True)
    .mul(100)
    .round(2)
)


# ------------------------------------------
# F. Final Dataset Information
# ------------------------------------------

print("\nFinal Shape:", stocks_df.shape)

print(
    "Number of Features:",
    stocks_df.shape[1]
)


# ------------------------------------------
# G. Save Dataset
# ------------------------------------------

stocks_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    "\nFeature engineered dataset saved to:",
    OUTPUT_PATH
)

stocks_df.head()


Columns containing missing values:

Trades                 114848
Deliverable Volume      16077
%Deliverble             16077
Price_SMA_200_Ratio      9751
SMA_200                  9751
SMA_100                  4851
Price_SMA_100_Ratio      4851
Price_SMA_50_Ratio       2401
SMA_50                   2401
Volatility_30            1470
Momentum_20               980
Volatility_20             980
SMA_20                    931
BB_Position               931
BB_Middle                 931
BB_Upper                  931
Volume_MA_20              931
Relative_Volume           931
BB_Lower                  931
Price_SMA_20_Ratio        931
RSI_14                    686
Return_Lag_10             539
Volatility_10             490
Momentum_10               490
SMA_10                    441
Price_SMA_10_Ratio        441
Return_Lag_5              294
Momentum_5                245
Close_Lag_5               245
Volatility_5              245
SMA_5                     196
Return_Lag_3              196
Pri

,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,Volatility_30,Volume_Change,Volume_MA_20,Relative_Volume,Year,Month,DayOfWeek,Quarter,Future_Return_1D,Target_Direction
0,2007-11-27,ADANIPORTS,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,...,NaN,NaN,NaN,NaN,2007,11,1,4,-0.071659,0.0
1,2007-11-28,ADANIPORTS,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,...,NaN,-0.832151,NaN,NaN,2007,11,2,4,-0.010851,0.0
2,2007-11-29,ADANIPORTS,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,...,NaN,0.118477,NaN,NaN,2007,11,3,4,0.042242,1.0
3,2007-11-30,ADANIPORTS,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,...,NaN,-0.100380,NaN,NaN,2007,11,4,4,0.051815,1.0
4,2007-12-03,ADANIPORTS,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,...,NaN,-0.354095,NaN,NaN,2007,12,0,4,0.074435,1.0
